# Modelo no supervisado — sismos de Santander

Este cuaderno agrupa los **municipios** de Santander según cómo se comportan los sismos.

- Lee el dataset ya limpio: `datos/processed/dataset_limpio.csv`.
- Usa la ventana **2006–2026** que eligió el equipo.
- Arma **k=4 grupos**: Bajo, Moderado, Alto y Muy alto.
- Guarda los resultados en esta misma carpeta.

Para correrlo: **Restart & Run All**. En Colab, sube primero `datos/processed/dataset_limpio.csv` (y `fallas_santander.geojson` si quieres incluir fallas).


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (silhouette_score, davies_bouldin_score,
                             calinski_harabasz_score, adjusted_rand_score)
import joblib

RUTA_EVENTOS = Path("datos/processed/dataset_limpio.csv")
RUTA_FALLAS = Path("datos/processed/fallas_santander.geojson")
CARPETA = Path("modelos/no_supervisado")

ANIO_INICIO = 2006   # ventana elegida por el equipo
MIN_EVENTOS = 15     # municipios con menos sismos no entran
K_FINAL = 4          # Bajo, Moderado, Alto, Muy alto
SEMILLA = 42
np.random.seed(SEMILLA)


## 1. Cargar los datos

Leemos solo el archivo procesado (nada de crudos). Ojo: 2014–2017 no tiene datos porque ese bloque no está descargado.


In [ ]:
df = pd.read_csv(RUTA_EVENTOS, parse_dates=["fecha"])

print("Tipos de magnitud en todo el dataset:")
print(df["tipo_mag"].value_counts().to_string())

# Nos quedamos con la ventana del equipo y con la magnitud Ml (casi todos los sismos vienen así).
df = df[df["fecha"].dt.year >= ANIO_INICIO].copy()
df = df[df["tipo_mag"] == "Ml"].copy()

print("\nEventos desde", ANIO_INICIO, ":", len(df))
print("Fechas:", df["fecha"].min().date(), "->", df["fecha"].max().date())
print("Municipios:", df["municipio"].nunique())


## 2. Variables por municipio

Resumimos cada municipio en 20 variables: cuántos sismos tiene, qué tan grandes, qué tan profundos, etc. Las tasas se calculan con los años que sí tienen datos.


In [ ]:
df["anio"] = df["fecha"].dt.year
anios = df["anio"].nunique()
df["es_m3"] = (df["mag"] >= 3).astype(int)
df["es_m4"] = (df["mag"] >= 4).astype(int)
df["energia"] = 10 ** (1.5 * df["mag"] + 4.8)   # energía liberada (regla de Gutenberg-Richter)
df["reciente"] = (df["fecha"] >= df["fecha"].max() - pd.DateOffset(years=5)).astype(int)
df["sup"] = (df["prof_km"] < 30).astype(int)
df["inter"] = df["prof_km"].between(30, 150).astype(int)
df["pro"] = (df["prof_km"] >= 150).astype(int)

def haversine(lat1, lon1, lat2, lon2):
    # Distancia en km entre dos puntos del planeta.
    R = 6371.0
    la1, lo1, la2, lo2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((la2 - la1) / 2) ** 2 + np.cos(la1) * np.cos(la2) * np.sin((lo2 - lo1) / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

NIDO_LAT, NIDO_LON = 6.8, -73.1   # nido sísmico de Bucaramanga (aprox.)

g = df.groupby("municipio")
F = pd.DataFrame({
    "n_sismos": g.size(),
    "sismos_x_anio": g.size() / anios,
    "mag_media": g["mag"].mean(),
    "mag_max": g["mag"].max(),
    "mag_std": g["mag"].std(),
    "mag_p90": g["mag"].quantile(0.90),
    "pct_m3_mas": g["es_m3"].mean(),
    "pct_m4_mas": g["es_m4"].mean(),
    "log_energia": np.log10(g["energia"].sum()),
    "prof_media": g["prof_km"].mean(),
    "prof_std": g["prof_km"].std(),
    "pct_superficial": g["sup"].mean(),
    "pct_intermedia": g["inter"].mean(),
    "pct_profunda": g["pro"].mean(),
    "rms_medio": g["rms_seg"].mean(),
    "gap_medio": g["gap_grados"].mean(),
    "fases_medias": g["n_fases"].mean(),
    "frac_reciente": g["reciente"].mean(),
    "lat_centro": g["lat"].mean(),
    "lon_centro": g["lon"].mean(),
})
F["dist_nido_km"] = haversine(F["lat_centro"], F["lon_centro"], NIDO_LAT, NIDO_LON)

F = F[F["n_sismos"] >= MIN_EVENTOS].copy()
F = F.fillna(F.median(numeric_only=True))
print("Municipios con", MIN_EVENTOS, "o más sismos:", len(F))
F.head()


## 3. Fallas geológicas (si están)

Si existe `fallas_santander.geojson`, agregamos la distancia del centro del municipio a la falla visible más cercana. Es una aproximación.


In [ ]:
if RUTA_FALLAS.exists():
    geo = json.loads(RUTA_FALLAS.read_text(encoding="utf-8"))
    puntos = []
    for f in geo["features"]:
        for parte in f["geometry"]["coordinates"]:
            puntos.extend(parte)
    puntos = np.array(puntos)

    def dist_min_falla(lat, lon):
        return float(haversine(lat, lon, puntos[:, 1], puntos[:, 0]).min())

    F["dist_falla_km"] = [dist_min_falla(la, lo) for la, lo in zip(F["lat_centro"], F["lon_centro"])]
    print("Fallas cargadas:", len(puntos), "puntos")
else:
    print("No hay archivo de fallas; seguimos sin esa variable")


## 4. Preparar las variables

Escalamos los números (para que ninguno pese más que otro por su tamaño) y resumimos con PCA.


In [ ]:
COLS_LOG = [c for c in ["n_sismos", "sismos_x_anio", "dist_nido_km", "dist_falla_km"] if c in F]
X = F.copy()
X[COLS_LOG] = np.log1p(X[COLS_LOG])

# Las coordenadas se quedan para el mapa, no entran al modelo.
COLS_MODELO = [c for c in X.columns if c not in ("lat_centro", "lon_centro")]

escalador = RobustScaler()
Xs = np.clip(escalador.fit_transform(X[COLS_MODELO]), -5, 5)

pca = PCA(n_components=0.90, random_state=SEMILLA)
Z = pca.fit_transform(Xs)
print("Variables del modelo:", len(COLS_MODELO))
print("Componentes PCA (90% de la información):", Z.shape[1])


## 5. Comparar modelos (evidencia)

Probamos KMeans, Ward y GMM con k de 2 a 8. Esto queda como evidencia para el jurado. Nosotros usamos **k=4 fijo** (lo acordó el equipo).


In [ ]:
def ajustar(nombre, k, datos):
    if nombre == "KMeans":
        return KMeans(n_clusters=k, n_init=10, random_state=SEMILLA).fit_predict(datos)
    if nombre == "Ward":
        return AgglomerativeClustering(n_clusters=k, linkage="ward").fit_predict(datos)
    return GaussianMixture(n_components=k, covariance_type="diag", n_init=3, random_state=SEMILLA).fit_predict(datos)

def estabilidad(nombre, k, datos, repes=10, frac=0.8):
    # ¿Los grupos se mantienen si quitamos el 20% de los municipios al azar?
    rng = np.random.default_rng(SEMILLA)
    base = ajustar(nombre, k, datos)
    aris = []
    for _ in range(repes):
        idx = rng.choice(len(datos), int(frac * len(datos)), replace=False)
        aris.append(adjusted_rand_score(base[idx], ajustar(nombre, k, datos[idx])))
    return float(np.mean(aris))

filas = []
for nombre in ["KMeans", "Ward", "GMM"]:
    for k in range(2, 9):
        lab = ajustar(nombre, k, Z)
        filas.append({
            "modelo": nombre, "k": k,
            "silueta": silhouette_score(Z, lab),
            "davies": davies_bouldin_score(Z, lab),
            "calinski": calinski_harabasz_score(Z, lab),
            "estabilidad": estabilidad(nombre, k, Z),
        })
comparacion = pd.DataFrame(filas)
comparacion.round(3)


## 6. Armar los grupos

El índice de actividad mezcla frecuencia, energía, tamaño máximo y superficialidad. Un municipio con pocos sismos pero **muy superficiales** también cuenta como peligroso.


In [ ]:
modelo_final = KMeans(n_clusters=K_FINAL, n_init=10, random_state=SEMILLA)
F["grupo_num"] = modelo_final.fit_predict(Z)

IND = ["sismos_x_anio", "log_energia", "mag_max", "pct_m4_mas", "pct_superficial"]
zs = (F[IND] - F[IND].mean()) / F[IND].std()
F["indice_actividad"] = zs.mean(axis=1)

# Los grupos se ordenan de menor a mayor actividad y se etiquetan.
orden = F.groupby("grupo_num")["indice_actividad"].mean().sort_values().index
nombres = ["Bajo", "Moderado", "Alto", "Muy alto"]
etiquetas = {g: nombres[i] for i, g in enumerate(orden)}
F["grupo"] = F["grupo_num"].map(etiquetas)

print("Municipios por grupo:")
print(F["grupo"].value_counts().to_string())

perfil = F.groupby("grupo")[COLS_MODELO].median().round(2)
perfil.insert(0, "municipios", F.groupby("grupo").size())
perfil.T


## 7. ¿Qué variables separan los grupos?

Usamos Kruskal-Wallis (una prueba estadística simple) y miramos qué variables cambian más entre grupos.


In [ ]:
filas_imp = []
for c in COLS_MODELO:
    muestras = [d[c].values for _, d in F.groupby("grupo")]
    if all(np.std(m) == 0 for m in muestras):
        continue
    H, p = stats.kruskal(*muestras)
    filas_imp.append((c, H / (len(F) - 1), p))
importancia = pd.DataFrame(filas_imp, columns=["variable", "eta2", "p"]).sort_values("eta2", ascending=False)
importancia.head(12).round(4)


## 8. Gráficos


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for grupo, d in F.groupby("grupo"):
    donde = [F.index.get_loc(m) for m in d.index]
    ax.scatter(Z[donde, 0], Z[donde, 1], s=60, label=grupo, alpha=.8)
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
ax.legend(title="Grupo"); ax.set_title("Municipios en el plano PCA")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 8))
if RUTA_FALLAS.exists():
    ax.scatter(puntos[:, 0], puntos[:, 1], s=1, c="lightgray", alpha=.5, label="Fallas")
for grupo, d in F.groupby("grupo"):
    ax.scatter(d["lon_centro"], d["lat_centro"], s=40, label=grupo, edgecolor="k", linewidth=.4)
ax.scatter(NIDO_LON, NIDO_LAT, marker="*", s=250, c="red", label="Nido de Bucaramanga (ref.)")
ax.set_xlabel("Longitud"); ax.set_ylabel("Latitud")
ax.legend(fontsize=8); ax.set_title("Municipios por grupo")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
importancia.head(10).iloc[::-1].plot.barh(x="variable", y="eta2", legend=False, color="#8e44ad", ax=ax)
ax.set_title("Variables que más separan los grupos")
plt.tight_layout(); plt.show()


## 9. Guardar resultados

Todo queda en `modelos/no_supervisado/`.


In [ ]:
CARPETA.mkdir(parents=True, exist_ok=True)
F[["grupo", "indice_actividad"]].to_csv(CARPETA / "grupo_por_municipio.csv", encoding="utf-8")
perfil.to_csv(CARPETA / "perfil_grupos.csv", encoding="utf-8")
comparacion.to_csv(CARPETA / "comparacion_modelos.csv", index=False, encoding="utf-8")
importancia.to_csv(CARPETA / "importancia_variables.csv", index=False, encoding="utf-8")
joblib.dump({"escalador": escalador, "pca": pca, "kmeans": modelo_final,
             "columnas": COLS_MODELO, "cols_log": COLS_LOG, "etiquetas": etiquetas},
            CARPETA / "modelo_no_supervisado.joblib")
print("Listo. Archivos guardados en", CARPETA)
print(F["grupo"].value_counts().to_string())


## 10. Para el pitch

- Encontramos dos tipos de peligro: **Muy alto** = mucha actividad (zona del nido sísmico, sismos profundos); **Alto** = pocos sismos pero superficiales (más dañinos).
- Límites honestos: falta 2014–2017, falta población/vivienda (DANE) y la distancia a fallas es aproximada.
